In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project="pm25-nicaragua-earth-engine")

print("Conexión con Earth Engine:", ee.Number(1).getInfo())

In [ ]:
from pathlib import Path

import requests
import rasterio

area_piloto = ee.Geometry.Rectangle([-86.31, 12.10, -86.22, 12.17])
id_escena = "LC08_L2SP_017052_20260202_20260217_02_T1"

escenas = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(area_piloto)
    .filterDate("2026-02-02", "2026-02-03")
    .filter(ee.Filter.eq("LANDSAT_PRODUCT_ID", id_escena))
)

if escenas.size().getInfo() == 0:
    raise ValueError(f"Earth Engine no encontró la escena {id_escena}.")

escena = ee.Image(escenas.first())
url = escena.select(["ST_B10", "QA_PIXEL"]).toUint16().getDownloadURL({
    "region": area_piloto,
    "scale": 30,
    "crs": "EPSG:32616",
    "format": "GEO_TIFF",
})

carpeta = Path("data/piloto")
carpeta.mkdir(parents=True, exist_ok=True)
ruta = carpeta / "managua_landsat_2026-02-02.tif"

respuesta = requests.get(url, timeout=(15, 120))
respuesta.raise_for_status()
ruta.write_bytes(respuesta.content)

with rasterio.open(ruta) as src:
    print(f"Bandas: {src.count} | Tamaño: {src.width} × {src.height}")
    print("CRS:", src.crs)

print("Guardado en:", ruta.resolve())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from rasterio.plot import plotting_extent

with rasterio.open(ruta) as src:
    temperatura_dn = src.read(1, masked=True)
    calidad = src.read(2, masked=True)
    extension = plotting_extent(src)

# Bits de relleno, borde de nube, cirros, nube, sombra y agua.
bits_excluir = sum(1 << bit for bit in (0, 1, 2, 3, 4, 7))

validos = (
    ~np.ma.getmaskarray(temperatura_dn)
    & ~np.ma.getmaskarray(calidad)
    & (temperatura_dn.data > 0)
    & ((calidad.data & bits_excluir) == 0)
)

if not validos.any():
    raise ValueError("El recorte quedó sin píxeles válidos.")

lst_c = np.full(temperatura_dn.shape, np.nan, dtype="float32")
lst_c[validos] = temperatura_dn.data[validos] * 0.00341802 + 149.0 - 273.15

p05, mediana, p95 = np.nanpercentile(lst_c, [5, 50, 95])
print(f"Píxeles válidos: {validos.sum():,} de {validos.size:,}")
print(f"P05: {p05:.1f} °C | Mediana: {mediana:.1f} °C | P95: {p95:.1f} °C")

fig, ax = plt.subplots(figsize=(8, 6))
mapa = ax.imshow(lst_c, extent=extension, cmap="inferno", vmin=p05, vmax=p95)
fig.colorbar(mapa, ax=ax, label="Temperatura superficial (°C)")
ax.set_title("Centro de Managua · 2 de febrero de 2026")
ax.set_axis_off()
plt.show()

In [ ]:
import pandas as pd

tamano_celda = 250

with rasterio.open(ruta) as src:
    transformacion = src.transform
    print("CRS:", src.crs)

filas, columnas = np.indices(lst_c.shape)
x, y = transformacion * (columnas + 0.5, filas + 0.5)

pixeles = pd.DataFrame({
    "col_celda": np.floor(x.ravel() / tamano_celda).astype(int),
    "fila_celda": np.floor(y.ravel() / tamano_celda).astype(int),
    "temperatura_c": lst_c.ravel(),
})

celdas = (
    pixeles.groupby(["col_celda", "fila_celda"], as_index=False)
    .agg(
        temperatura_mediana_c=("temperatura_c", "median"),
        n_pixeles_recorte=("temperatura_c", "size"),
        n_pixeles_validos=("temperatura_c", "count"),
    )
)

celdas["fraccion_valida"] = (
    celdas["n_pixeles_validos"] / celdas["n_pixeles_recorte"]
)
celdas["id_celda"] = (
    celdas["col_celda"].astype(str) + "_" + celdas["fila_celda"].astype(str)
)
celdas["fecha"] = pd.Timestamp("2026-02-02")

salida = ruta.parent / "managua_celdas_250m_2026-02-02.csv"
celdas.to_csv(salida, index=False)

print("Celdas en el recorte:", len(celdas))
print("Celdas con temperatura:", celdas["temperatura_mediana_c"].notna().sum())
print("Archivo:", salida.resolve())

celdas.head()

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, mapping

url_limites = "https://www.geoboundaries.org/api/current/gbOpen/NIC/ADM2/"
respuesta = requests.get(url_limites, timeout=60)
respuesta.raise_for_status()

municipios = gpd.read_file(
    respuesta.json()["gjDownloadURL"]
).to_crs("EPSG:4326")

punto_centro = Point(-86.2730, 12.1364)
managua = municipios.loc[municipios.geometry.covers(punto_centro)].copy()

if len(managua) != 1:
    raise ValueError(f"El punto coincidió con {len(managua)} municipios.")

area_managua = ee.Geometry(mapping(managua.geometry.iloc[0]))

print("Nombre en la fuente:", managua["shapeName"].iloc[0])
print("Límites:", managua.total_bounds.round(4))

In [ ]:
landsat = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(
    ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
)

termicas_managua = (
    landsat.filterBounds(area_managua)
    .filter(ee.Filter.eq("PROCESSING_LEVEL", "L2SP"))
)

for anio in range(2023, 2027):
    temporada = termicas_managua.filterDate(f"{anio}-01-01", f"{anio}-05-01")
    print(f"Enero–abril de {anio}: {temporada.size().getInfo()} escenas")

In [ ]:
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from rasterio.features import geometry_mask
from rasterio.io import MemoryFile
from shapely.geometry import mapping

candidatas = (
    termicas_managua.filterDate("2026-01-01", "2026-05-01")
    .filter(ee.Filter.eq("WRS_PATH", 17))
    .filter(ee.Filter.eq("WRS_ROW", 52))
    .sort("CLOUD_COVER")
    .limit(3)
    .aggregate_array("system:id")
    .getInfo()
)

if not candidatas:
    raise ValueError("No encontré escenas de la trayectoria 17/52 para el piloto.")

geometria_utm = mapping(managua.to_crs("EPSG:32616").geometry.iloc[0])
bandas = ["ST_B10", "QA_PIXEL"]
bits_excluir = sum(1 << bit for bit in (0, 1, 2, 3, 4, 7))
tablas = []

for id_escena in candidatas:
    escena = ee.Image(id_escena)
    fecha = escena.date().format("YYYY-MM-dd").getInfo()

    url = escena.select(bandas).toUint16().getDownloadURL({
        "region": area_managua.bounds(),
        "scale": 30,
        "crs": "EPSG:32616",
        "format": "GEO_TIFF",
    })
    respuesta = requests.get(url, timeout=(15, 120))
    respuesta.raise_for_status()

    with MemoryFile(BytesIO(respuesta.content)) as archivo:
        with archivo.open() as src:
            termica = src.read(1, masked=True)
            qa = src.read(2, masked=True)
            transformacion = src.transform

    dentro = geometry_mask(
        [geometria_utm],
        out_shape=termica.shape,
        transform=transformacion,
        invert=True,
    )

    validos = (
        dentro
        & ~np.ma.getmaskarray(termica)
        & ~np.ma.getmaskarray(qa)
        & (termica.data > 0)
        & ((qa.data & bits_excluir) == 0)
    )

    temperatura = np.full(termica.shape, np.nan, dtype="float32")
    temperatura[validos] = (
        termica.data[validos] * 0.00341802 + 149.0 - 273.15
    )

    filas, columnas = np.nonzero(dentro)
    x, y = transformacion * (columnas + 0.5, filas + 0.5)

    pixeles = pd.DataFrame({
        "col_celda": np.floor(x / 250).astype(int),
        "fila_celda": np.floor(y / 250).astype(int),
        "temperatura_c": temperatura[filas, columnas],
    })

    resumen = (
        pixeles.groupby(["col_celda", "fila_celda"], as_index=False)
        .agg(
            temperatura_mediana_c=("temperatura_c", "median"),
            n_pixeles_municipio=("temperatura_c", "size"),
            n_pixeles_validos=("temperatura_c", "count"),
        )
    )
    resumen["fraccion_valida"] = (
        resumen["n_pixeles_validos"] / resumen["n_pixeles_municipio"]
    )
    resumen["id_celda"] = (
        resumen["col_celda"].astype(str) + "_" + resumen["fila_celda"].astype(str)
    )
    resumen["fecha"] = fecha
    resumen["id_escena"] = id_escena
    tablas.append(resumen)

    print(
        f"{fecha} | {validos.sum() / dentro.sum():.1%} de píxeles válidos "
        f"| {resumen['temperatura_mediana_c'].notna().sum():,} celdas con temperatura"
    )

panel_piloto = pd.concat(tablas, ignore_index=True)
salida = Path("data/piloto/managua_municipio_250m_tres_escenas.csv")
salida.parent.mkdir(parents=True, exist_ok=True)
panel_piloto.to_csv(salida, index=False)

print("Archivo:", salida.resolve())

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import box

datos = panel_piloto.copy()
total_fechas = datos["fecha"].nunique()

if total_fechas != 3 or datos.duplicated(["id_celda", "fecha"]).any():
    raise ValueError("El piloto requiere tres fechas y una fila por celda y fecha.")

elegibles = datos.loc[
    datos["fraccion_valida"].ge(0.70)
    & datos["temperatura_mediana_c"].notna()
].copy()

fechas_por_celda = elegibles.groupby("id_celda")["fecha"].nunique()
ids_comunes = fechas_por_celda[fechas_por_celda.eq(total_fechas)].index
comparables = elegibles.loc[elegibles["id_celda"].isin(ids_comunes)].copy()

if comparables.empty:
    raise ValueError("No quedaron celdas con cobertura suficiente en las tres fechas.")

umbrales = comparables.groupby("fecha")["temperatura_mediana_c"].quantile(0.90)
comparables["p90_fecha_c"] = comparables["fecha"].map(umbrales)
comparables["caliente"] = comparables["temperatura_mediana_c"].ge(
    comparables["p90_fecha_c"]
)

recurrencia = (
    comparables.groupby(["id_celda", "col_celda", "fila_celda"], as_index=False)
    .agg(
        fechas_calientes=("caliente", "sum"),
        fechas_validas=("fecha", "nunique"),
    )
)

print("Celdas comparables:", len(recurrencia))
print("\nP90 por fecha (°C):")
print(umbrales.round(2))
print("\nCeldas por número de fechas calientes:")
print(recurrencia["fechas_calientes"].value_counts().sort_index())

geometrias = [
    box(col * 250, fila * 250, (col + 1) * 250, (fila + 1) * 250)
    for col, fila in zip(recurrencia["col_celda"], recurrencia["fila_celda"])
]
mapa = gpd.GeoDataFrame(recurrencia, geometry=geometrias, crs="EPSG:32616")

fig, ax = plt.subplots(figsize=(9, 8))
managua.to_crs(mapa.crs).plot(
    ax=ax, color="#eeeeee", edgecolor="#666666", linewidth=0.8
)
mapa.plot(
    ax=ax,
    column="fechas_calientes",
    cmap="YlOrRd",
    categorical=True,
    legend=True,
    legend_kwds={"title": "Fechas en el P90 o por encima"},
)

ax.set_title("Managua · Recurrencia de calor superficial en tres fechas")
ax.set_axis_off()
plt.show()

In [ ]:
temperaturas = comparables.pivot(
    index="id_celda",
    columns="fecha",
    values="temperatura_mediana_c",
).sort_index(axis=1)

print("Temperatura superficial por fecha (°C)")
display(temperaturas.quantile([0.10, 0.50, 0.90, 0.99]).T.round(2))

diferencias = pd.DataFrame({
    "22 mar − 2 feb": temperaturas["2026-03-22"] - temperaturas["2026-02-02"],
    "22 mar − 10 feb": temperaturas["2026-03-22"] - temperaturas["2026-02-10"],
})

resumen = diferencias.quantile([0.10, 0.50, 0.90]).T
resumen.columns = ["P10_diferencia_c", "mediana_diferencia_c", "P90_diferencia_c"]
resumen["pct_celdas_mas_calientes_en_marzo"] = diferencias.gt(0).mean() * 100

display(resumen.round(2))

In [ ]:
registros = []

for id_escena in panel_piloto["id_escena"].drop_duplicates():
    imagen = ee.Image(id_escena)
    metadatos = imagen.toDictionary([
        "LANDSAT_PRODUCT_ID",
        "SPACECRAFT_ID",
        "CLOUD_COVER",
        "SUN_ELEVATION",
    ]).getInfo()

    registros.append({
        "fecha_hora_utc": imagen.date().format("YYYY-MM-dd HH:mm:ss").getInfo(),
        **metadatos,
    })

display(pd.DataFrame(registros).sort_values("fecha_hora_utc"))

In [ ]:
# Ejecutá esto mientras panel_piloto contiene las tres escenas originales.
fechas_base = dict(
    panel_piloto[["fecha", "id_escena"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

trayectoria = (
    termicas_managua.filter(ee.Filter.eq("WRS_PATH", 17))
    .filter(ee.Filter.eq("WRS_ROW", 52))
)

seleccion = {}

for mes in range(1, 5):
    inicio = ee.Date.fromYMD(2026, mes, 1)
    escenas_mes = trayectoria.filterDate(inicio, inicio.advance(1, "month"))

    fechas_mes = {
        fecha: id_escena
        for fecha, id_escena in fechas_base.items()
        if fecha.startswith(f"2026-{mes:02d}-")
    }

    ids_ordenados = (
        escenas_mes.sort("CLOUD_COVER")
        .aggregate_array("system:id")
        .getInfo()
    )

    for id_escena in ids_ordenados:
        if len(fechas_mes) >= 2:
            break

        fecha = ee.Image(id_escena).date().format("YYYY-MM-dd").getInfo()
        fechas_mes.setdefault(fecha, id_escena)

    seleccion.update(fechas_mes)

candidatas_ampliadas = [seleccion[fecha] for fecha in sorted(seleccion)]

for fecha in sorted(seleccion):
    print(fecha, seleccion[fecha])

print("Total de fechas:", len(candidatas_ampliadas))

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from rasterio.features import geometry_mask
from rasterio.io import MemoryFile
from shapely.geometry import mapping

candidatas = candidatas_ampliadas

if len(candidatas) != 8 or len(set(candidatas)) != 8:
    raise ValueError("Esperaba ocho escenas distintas en candidatas_ampliadas.")

geometria_utm = mapping(managua.to_crs("EPSG:32616").geometry.iloc[0])
bandas = ["ST_B10", "QA_PIXEL"]
bits_excluir = sum(1 << bit for bit in (0, 1, 2, 3, 4, 7))
tablas = []

for id_escena in candidatas:
    escena = ee.Image(id_escena)
    fecha = escena.date().format("YYYY-MM-dd").getInfo()

    url = escena.select(bandas).toUint16().getDownloadURL({
        "region": area_managua.bounds(),
        "scale": 30,
        "crs": "EPSG:32616",
        "format": "GEO_TIFF",
    })

    respuesta = requests.get(url, timeout=(15, 120))
    respuesta.raise_for_status()

    with MemoryFile(respuesta.content) as archivo:
        with archivo.open() as src:
            termica = src.read(1, masked=True)
            qa = src.read(2, masked=True)
            transformacion = src.transform

    dentro = geometry_mask(
        [geometria_utm],
        out_shape=termica.shape,
        transform=transformacion,
        invert=True,
    )

    validos = (
        dentro
        & ~np.ma.getmaskarray(termica)
        & ~np.ma.getmaskarray(qa)
        & (termica.data > 0)
        & ((qa.data & bits_excluir) == 0)
    )

    temperatura = np.full(termica.shape, np.nan, dtype="float32")
    temperatura[validos] = (
        termica.data[validos] * 0.00341802 + 149.0 - 273.15
    )

    filas, columnas = np.nonzero(dentro)
    x, y = transformacion * (columnas + 0.5, filas + 0.5)

    pixeles = pd.DataFrame({
        "col_celda": np.floor(x / 250).astype(int),
        "fila_celda": np.floor(y / 250).astype(int),
        "temperatura_c": temperatura[filas, columnas],
    })

    resumen = (
        pixeles.groupby(["col_celda", "fila_celda"], as_index=False)
        .agg(
            temperatura_mediana_c=("temperatura_c", "median"),
            n_pixeles_municipio=("temperatura_c", "size"),
            n_pixeles_validos=("temperatura_c", "count"),
        )
    )

    resumen["fraccion_valida"] = (
        resumen["n_pixeles_validos"] / resumen["n_pixeles_municipio"]
    )
    resumen["id_celda"] = (
        resumen["col_celda"].astype(str)
        + "_"
        + resumen["fila_celda"].astype(str)
    )
    resumen["fecha"] = fecha
    resumen["id_escena"] = id_escena
    tablas.append(resumen)

    cobertura = validos.sum() / dentro.sum()
    celdas_con_temperatura = resumen["temperatura_mediana_c"].notna().sum()

    print(
        f"{fecha} | {cobertura:.1%} de píxeles válidos "
        f"| {celdas_con_temperatura:,} celdas con temperatura"
    )

panel_piloto = pd.concat(tablas, ignore_index=True)

salida = Path("data/piloto/managua_municipio_250m_enero_abril_2026.csv")
salida.parent.mkdir(parents=True, exist_ok=True)
panel_piloto.to_csv(salida, index=False)

print(f"\nFechas procesadas: {panel_piloto['fecha'].nunique()}")
print("Archivo:", salida.resolve())

In [ ]:
def agregar_cobertura_local(imagen):
    termica = imagen.select("ST_B10")
    qa = imagen.select("QA_PIXEL")

    valido = (
        termica.gt(0)
        .And(qa.bitwiseAnd(bits_excluir).eq(0))
        .unmask(0, False)
        .rename("valido")
    )

    cobertura = valido.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area_managua,
        crs="EPSG:32616",
        scale=90,
        maxPixels=1_000_000,
    ).get("valido")

    return imagen.set("cobertura_local", cobertura)


coleccion = trayectoria.filterDate("2026-01-01", "2026-05-01")
filas = []

for mes in range(1, 5):
    inicio = ee.Date.fromYMD(2026, mes, 1)

    mejores = (
        coleccion.filterDate(inicio, inicio.advance(1, "month"))
        .map(agregar_cobertura_local)
        .sort("cobertura_local", False)
        .limit(3)
    )

    ids = mejores.aggregate_array("system:id").getInfo()
    coberturas = mejores.aggregate_array("cobertura_local").getInfo()

    for id_escena, cobertura in zip(ids, coberturas):
        filas.append({
            "mes": mes,
            "id_escena": id_escena,
            "cobertura_local_pct": round(100 * cobertura, 1),
        })

ranking_local = pd.DataFrame(filas)
display(ranking_local)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from rasterio.features import geometry_mask
from rasterio.io import MemoryFile
from shapely.geometry import mapping

seleccion_principal = ranking_local.loc[
    ranking_local["cobertura_local_pct"].ge(70)
].sort_values(["mes", "id_escena"])

candidatas = seleccion_principal["id_escena"].drop_duplicates().tolist()

if len(candidatas) != 5:
    raise ValueError(
        f"Esperaba cinco escenas con cobertura local ≥ 70 %. "
        f"Encontré {len(candidatas)}."
    )

geometria_utm = mapping(managua.to_crs("EPSG:32616").geometry.iloc[0])
region_descarga = area_managua.bounds()
bandas = ["ST_B10", "QA_PIXEL"]
bits_excluir = sum(1 << bit for bit in (0, 1, 2, 3, 4, 7))
tamano_celda = 250
tablas = []

for id_escena in candidatas:
    escena = ee.Image(id_escena)
    fecha = escena.date().format("YYYY-MM-dd").getInfo()

    url = escena.select(bandas).toUint16().getDownloadURL({
        "region": region_descarga,
        "scale": 30,
        "crs": "EPSG:32616",
        "format": "GEO_TIFF",
    })

    respuesta = requests.get(url, timeout=(15, 120))
    respuesta.raise_for_status()

    with MemoryFile(respuesta.content) as archivo:
        with archivo.open() as src:
            termica = src.read(1, masked=True)
            qa = src.read(2, masked=True)
            transformacion = src.transform

    dentro = geometry_mask(
        [geometria_utm],
        out_shape=termica.shape,
        transform=transformacion,
        invert=True,
    )

    validos = (
        dentro
        & ~np.ma.getmaskarray(termica)
        & ~np.ma.getmaskarray(qa)
        & (termica.data > 0)
        & ((qa.data & bits_excluir) == 0)
    )

    temperatura = np.full(termica.shape, np.nan, dtype="float32")
    temperatura[validos] = (
        termica.data[validos] * 0.00341802 + 149.0 - 273.15
    )

    filas, columnas = np.nonzero(dentro)
    x, y = transformacion * (columnas + 0.5, filas + 0.5)

    pixeles = pd.DataFrame({
        "col_celda": np.floor(x / tamano_celda).astype(int),
        "fila_celda": np.floor(y / tamano_celda).astype(int),
        "temperatura_c": temperatura[filas, columnas],
    })

    resumen = (
        pixeles.groupby(["col_celda", "fila_celda"], as_index=False)
        .agg(
            temperatura_mediana_c=("temperatura_c", "median"),
            n_pixeles_municipio=("temperatura_c", "size"),
            n_pixeles_validos=("temperatura_c", "count"),
        )
    )

    resumen["fraccion_valida"] = (
        resumen["n_pixeles_validos"] / resumen["n_pixeles_municipio"]
    )
    resumen["id_celda"] = (
        resumen["col_celda"].astype(str)
        + "_"
        + resumen["fila_celda"].astype(str)
    )
    resumen["fecha"] = fecha
    resumen["id_escena"] = id_escena
    tablas.append(resumen)

    cobertura = validos.sum() / dentro.sum()
    celdas_con_temperatura = resumen["temperatura_mediana_c"].notna().sum()
    print(
        f"{fecha} | {cobertura:.1%} de píxeles válidos "
        f"| {celdas_con_temperatura:,} celdas con temperatura"
    )

panel_principal = (
    pd.concat(tablas, ignore_index=True)
    .sort_values(["fecha", "col_celda", "fila_celda"])
    .reset_index(drop=True)
)

salida_principal = Path(
    "data/piloto/managua_municipio_250m_cinco_escenas_2026.csv"
)
salida_principal.parent.mkdir(parents=True, exist_ok=True)
panel_principal.to_csv(salida_principal, index=False)

print(f"\nFechas procesadas: {panel_principal['fecha'].nunique()}")
print("Archivo:", salida_principal.resolve())

In [ ]:
from pathlib import Path

import pandas as pd

carpeta = Path("data/piloto")

# Cuatro de las cinco escenas ya están en este CSV.
panel_ocho = pd.read_csv(
    carpeta / "managua_municipio_250m_enero_abril_2026.csv",
    dtype={"fecha": "string"},
)

# El bucle guardó aquí cada escena que terminó antes del error.
panel_parcial = pd.concat(tablas, ignore_index=True)

# Usamos el listado de cinco escenas definido antes del error.
panel_principal = pd.concat([panel_ocho, panel_parcial], ignore_index=True)
panel_principal = (
    panel_principal.loc[panel_principal["id_escena"].isin(candidatas)]
    .drop_duplicates(subset=["id_escena", "id_celda"], keep="last")
    .sort_values(["fecha", "col_celda", "fila_celda"])
    .reset_index(drop=True)
)

faltantes = set(candidatas) - set(panel_principal["id_escena"])
if faltantes:
    raise ValueError(f"Faltan escenas en los resultados disponibles: {sorted(faltantes)}")

salida_principal = carpeta / "managua_municipio_250m_cinco_escenas_2026.csv"
panel_principal.to_csv(salida_principal, index=False)

print("Fechas recuperadas:", panel_principal["fecha"].nunique())
print(panel_principal.groupby("fecha")["temperatura_mediana_c"].count())
print("Archivo:", salida_principal.resolve())

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

datos = panel_principal.copy()
n_fechas = datos["fecha"].nunique()

if n_fechas != 5 or datos.duplicated(["id_celda", "fecha"]).any():
    raise ValueError("Esperaba cinco fechas y una fila por celda y fecha.")

validas = datos.loc[
    datos["fraccion_valida"].ge(0.70)
    & datos["temperatura_mediana_c"].notna()
].copy()

fechas_por_celda = validas.groupby("id_celda")["fecha"].nunique()
ids_comunes = fechas_por_celda.index[fechas_por_celda.eq(n_fechas)]
comparables_principal = validas.loc[validas["id_celda"].isin(ids_comunes)].copy()

if comparables_principal.empty:
    raise ValueError("No quedaron celdas con cobertura suficiente en las cinco fechas.")

p90 = (
    comparables_principal.groupby("fecha")["temperatura_mediana_c"]
    .quantile(0.90)
)
comparables_principal["p90_fecha_c"] = comparables_principal["fecha"].map(p90)
comparables_principal["caliente"] = comparables_principal["temperatura_mediana_c"].ge(
    comparables_principal["p90_fecha_c"]
)

recurrencia_principal = (
    comparables_principal
    .groupby(["id_celda", "col_celda", "fila_celda"], as_index=False)
    .agg(
        fechas_calientes=("caliente", "sum"),
        fechas_validas=("fecha", "nunique"),
    )
)

print("Celdas comparables:", len(recurrencia_principal))
print("\nP90 por fecha (°C):")
print(p90.round(2))
print("\nCeldas por número de fechas calientes:")
print(
    recurrencia_principal["fechas_calientes"]
    .value_counts()
    .reindex(range(n_fechas + 1), fill_value=0)
    .sort_index()
)

geometrias = [
    box(col * 250, fila * 250, (col + 1) * 250, (fila + 1) * 250)
    for col, fila in zip(
        recurrencia_principal["col_celda"], recurrencia_principal["fila_celda"]
    )
]
mapa_principal = gpd.GeoDataFrame(
    recurrencia_principal, geometry=geometrias, crs="EPSG:32616"
)

fig, ax = plt.subplots(figsize=(9, 8))
managua.to_crs(mapa_principal.crs).plot(
    ax=ax, color="#eeeeee", edgecolor="#666666", linewidth=0.8
)
mapa_principal.plot(
    ax=ax,
    column="fechas_calientes",
    cmap=plt.get_cmap("YlOrRd", n_fechas + 1),
    vmin=-0.5,
    vmax=n_fechas + 0.5,
    legend=True,
    legend_kwds={
        "label": "Fechas con temperatura en el P90 o superior",
        "ticks": list(range(n_fechas + 1)),
        "shrink": 0.75,
    },
)

ax.set_title("Managua · Recurrencia de calor superficial\nCinco fechas de enero a marzo de 2026")
ax.set_axis_off()

carpeta = Path("data/piloto")
fig.savefig(carpeta / "managua_recurrencia_cinco_fechas_2026.png",
            dpi=180, bbox_inches="tight")
recurrencia_principal.to_csv(
    carpeta / "managua_recurrencia_cinco_fechas_2026.csv", index=False
)
plt.show()

In [ ]:
from pathlib import Path

import pandas as pd

carpeta = Path("data/inventario_cobertura")
carpeta.mkdir(parents=True, exist_ok=True)

archivos_mensuales = []

for anio in range(2023, 2027):
    for mes in range(1, 5):
        archivo = carpeta / f"cobertura_{anio}_{mes:02d}.csv"
        archivos_mensuales.append(archivo)

        if archivo.exists():
            continue

        inicio = ee.Date.fromYMD(anio, mes, 1)
        escenas_mes = (
            trayectoria.filterDate(inicio, inicio.advance(1, "month"))
            .map(agregar_cobertura_local)
        )

        # Una sola consulta por mes para mantener juntos el ID y su cobertura.
        pares = escenas_mes.reduceColumns(
            reducer=ee.Reducer.toList(2),
            selectors=["system:id", "cobertura_local"],
        ).get("list").getInfo()

        mensual = pd.DataFrame(
            pares or [], columns=["id_escena", "fraccion_cobertura"]
        )
        mensual["anio"] = anio
        mensual["mes"] = mes
        mensual.to_csv(archivo, index=False)

        print(f"{anio}-{mes:02d} | {len(mensual)} escenas evaluadas")

inventario = pd.concat(
    [pd.read_csv(archivo) for archivo in archivos_mensuales],
    ignore_index=True,
)

aptas = inventario.loc[inventario["fraccion_cobertura"].ge(0.70)].copy()
seleccion_multianual = (
    aptas.sort_values(
        ["anio", "mes", "fraccion_cobertura", "id_escena"],
        ascending=[True, True, False, True],
    )
    .groupby(["anio", "mes"], sort=False)
    .head(2)
    .reset_index(drop=True)
)

tabla_meses = (
    seleccion_multianual.groupby(["anio", "mes"]).size()
    .unstack(fill_value=0)
    .reindex(index=range(2023, 2027), columns=range(1, 5), fill_value=0)
    .rename(columns={1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril"})
)

seleccion_multianual.to_csv(carpeta / "escenas_seleccionadas.csv", index=False)
display(tabla_meses)

In [ ]:
from pathlib import Path

import pandas as pd

carpeta_todas = Path("data/inventario_todas_trayectorias")
carpeta_todas.mkdir(parents=True, exist_ok=True)

campos = ["system:id", "system:time_start", "WRS_PATH", "WRS_ROW", "cobertura_local"]
columnas = ["id_escena", "tiempo_ms", "trayectoria", "fila_wrs", "fraccion_cobertura"]
archivos = []

for anio in range(2023, 2027):
    for mes in range(1, 5):
        archivo = carpeta_todas / f"cobertura_{anio}_{mes:02d}.csv"
        archivos.append(archivo)

        if archivo.exists():
            continue

        inicio = ee.Date.fromYMD(anio, mes, 1)
        escenas_mes = (
            termicas_managua.filterDate(inicio, inicio.advance(1, "month"))
            .map(agregar_cobertura_local)
        )

        registros = escenas_mes.reduceColumns(
            reducer=ee.Reducer.toList(len(campos)),
            selectors=campos,
        ).get("list").getInfo()

        mensual = pd.DataFrame(registros or [], columns=columnas)
        mensual["anio"] = anio
        mensual["mes"] = mes
        mensual.to_csv(archivo, index=False)
        print(f"{anio}-{mes:02d} | {len(mensual)} escenas evaluadas")

inventario_todas = pd.concat(
    [pd.read_csv(archivo) for archivo in archivos],
    ignore_index=True,
)

inventario_todas["fecha"] = (
    pd.to_datetime(inventario_todas["tiempo_ms"], unit="ms", utc=True)
    .dt.strftime("%Y-%m-%d")
)

# Si hay varias escenas en una fecha, conservamos la de mayor cobertura.
mejor_por_fecha = (
    inventario_todas.sort_values("fraccion_cobertura", ascending=False)
    .drop_duplicates("fecha")
)

fechas_aptas = mejor_por_fecha.loc[
    mejor_por_fecha["fraccion_cobertura"].ge(0.70)
].copy()

tabla_todas = (
    fechas_aptas.groupby(["anio", "mes"]).size()
    .unstack(fill_value=0)
    .reindex(index=range(2023, 2027), columns=range(1, 5), fill_value=0)
    .rename(columns={1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril"})
)

display(tabla_todas)

In [ ]:
import pandas as pd

# ESA WorldCover 2021. La clase 80 identifica agua permanente.
cobertura_suelo = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
tierra = cobertura_suelo.neq(80)

# Reutilizamos las 16 consultas mensuales y guardamos los resultados
# en otra carpeta, porque cambió el denominador.
carpeta_tierra = Path("data/inventario_cobertura_terrestre")
carpeta_tierra.mkdir(parents=True, exist_ok=True)


def agregar_cobertura_terrestre(imagen):
    termica = imagen.select("ST_B10")
    qa = imagen.select("QA_PIXEL")

    valido = (
        termica.gt(0)
        .And(qa.bitwiseAnd(bits_excluir).eq(0))
        .unmask(0, False)
        .updateMask(tierra)
        .rename("valido")
    )

    cobertura = valido.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area_managua,
        crs="EPSG:32616",
        scale=90,
        maxPixels=1_000_000,
    ).get("valido")

    return imagen.set("cobertura_terrestre", cobertura)


archivos_tierra = []

for anio in range(2023, 2027):
    for mes in range(1, 5):
        archivo = carpeta_tierra / f"cobertura_{anio}_{mes:02d}.csv"
        archivos_tierra.append(archivo)

        if archivo.exists():
            continue

        inicio = ee.Date.fromYMD(anio, mes, 1)
        escenas = (
            termicas_managua.filterDate(inicio, inicio.advance(1, "month"))
            .map(agregar_cobertura_terrestre)
        )

        registros = escenas.reduceColumns(
            reducer=ee.Reducer.toList(3),
            selectors=["system:id", "system:time_start", "cobertura_terrestre"],
        ).get("list").getInfo()

        mensual = pd.DataFrame(
            registros or [],
            columns=["id_escena", "tiempo_ms", "fraccion_terrestre"],
        )
        mensual["anio"] = anio
        mensual["mes"] = mes
        mensual.to_csv(archivo, index=False)
        print(f"{anio}-{mes:02d} | {len(mensual)} escenas evaluadas")

inventario_tierra = pd.concat(
    [pd.read_csv(archivo) for archivo in archivos_tierra],
    ignore_index=True,
)

inventario_tierra["fecha"] = pd.to_datetime(
    inventario_tierra["tiempo_ms"], unit="ms", utc=True
).dt.strftime("%Y-%m-%d")

mejor_tierra_por_fecha = (
    inventario_tierra.sort_values("fraccion_terrestre", ascending=False)
    .drop_duplicates("fecha")
)

tabla_tierra = (
    mejor_tierra_por_fecha.loc[
        mejor_tierra_por_fecha["fraccion_terrestre"].ge(0.70)
    ]
    .groupby(["anio", "mes"]).size()
    .unstack(fill_value=0)
    .reindex(index=range(2023, 2027), columns=range(1, 5), fill_value=0)
    .rename(columns={1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril"})
)

display(tabla_tierra)

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from rasterio.features import geometry_mask
from rasterio.io import MemoryFile
from shapely.geometry import mapping

seleccion = mejor_tierra_por_fecha.loc[
    mejor_tierra_por_fecha["fraccion_terrestre"].ge(0.70)
].sort_values("fecha")

if seleccion.empty:
    raise ValueError("No hay escenas con cobertura terrestre ≥ 70 %.")

carpeta = Path("data/celdas_por_escena_2023_2026")
carpeta.mkdir(parents=True, exist_ok=True)

geometria_utm = mapping(managua.to_crs("EPSG:32616").geometry.iloc[0])
region = area_managua.bounds()
bits_excluir = sum(1 << bit for bit in (0, 1, 2, 3, 4, 7))
mascara_tierra = tierra.unmask(0).rename("TIERRA")


def descargar_geotiff(imagen, fecha):
    for intento in range(4):
        try:
            url = imagen.getDownloadURL({
                "region": region,
                "scale": 30,
                "crs": "EPSG:32616",
                "format": "GEO_TIFF",
            })
            respuesta = requests.get(url, timeout=(15, 120))
            respuesta.raise_for_status()
            return respuesta.content

        except requests.HTTPError as error:
            if error.response.status_code not in {429, 500, 502, 503, 504}:
                raise

        except (requests.ConnectionError, requests.Timeout):
            pass

        if intento < 3:
            espera = 2 ** (intento + 1)
            print(f"{fecha} | reintento de descarga en {espera} s")
            time.sleep(espera)

    raise RuntimeError(f"{fecha}: descarga pendiente tras cuatro intentos.")


archivos = []
pendientes = []

for escena in seleccion.itertuples(index=False):
    archivo = carpeta / f"celdas_{escena.fecha}.csv"

    if archivo.exists():
        archivos.append(archivo)
        print(f"{escena.fecha} | reutilizada")
        continue

    imagen = (
        ee.Image(escena.id_escena)
        .select(["ST_B10", "QA_PIXEL"])
        .addBands(mascara_tierra)
        .toUint16()
    )

    try:
        contenido = descargar_geotiff(imagen, escena.fecha)
    except RuntimeError as error:
        print(error)
        pendientes.append(escena.fecha)
        continue

    with MemoryFile(contenido) as memoria:
        with memoria.open() as src:
            termica, qa, es_tierra = src.read()
            transformacion = src.transform

    dentro = geometry_mask(
        [geometria_utm],
        out_shape=termica.shape,
        transform=transformacion,
        invert=True,
    )

    filas, columnas = np.nonzero(dentro & (es_tierra == 1))
    x, y = transformacion * (columnas + 0.5, filas + 0.5)

    dn = termica[filas, columnas]
    calidad = qa[filas, columnas]
    validos = (dn > 0) & ((calidad & bits_excluir) == 0)

    temperatura = np.full(dn.shape, np.nan, dtype="float32")
    temperatura[validos] = dn[validos] * 0.00341802 + 149.0 - 273.15

    pixeles = pd.DataFrame({
        "col_celda": np.floor(x / 250).astype(int),
        "fila_celda": np.floor(y / 250).astype(int),
        "temperatura_c": temperatura,
    })

    resumen = (
        pixeles.groupby(["col_celda", "fila_celda"], as_index=False)
        .agg(
            temperatura_mediana_c=("temperatura_c", "median"),
            n_pixeles_tierra=("temperatura_c", "size"),
            n_pixeles_validos=("temperatura_c", "count"),
        )
    )

    resumen["fraccion_valida"] = (
        resumen["n_pixeles_validos"] / resumen["n_pixeles_tierra"]
    )
    resumen["id_celda"] = (
        resumen["col_celda"].astype(str) + "_" + resumen["fila_celda"].astype(str)
    )
    resumen["fecha"] = escena.fecha
    resumen["id_escena"] = escena.id_escena
    resumen["anio"] = escena.anio
    resumen["mes"] = escena.mes

    temporal = archivo.with_suffix(".part")
    resumen.to_csv(temporal, index=False)
    temporal.replace(archivo)
    archivos.append(archivo)

    print(f"{escena.fecha} | {validos.mean():.1%} de tierra con datos válidos")

if archivos:
    panel_multianual = pd.concat(
        [pd.read_csv(archivo) for archivo in archivos],
        ignore_index=True,
    )
    salida = carpeta / "managua_panel_250m_2023_2026.csv"
    panel_multianual.to_csv(salida, index=False)
    print(f"\nFechas guardadas: {panel_multianual['fecha'].nunique()} de {len(seleccion)}")
    print("Archivo:", salida.resolve())

print("Fechas pendientes:", pendientes)

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

datos = panel_multianual.copy()
n_fechas = datos["fecha"].nunique()

if n_fechas != 18 or datos.duplicated(["id_celda", "fecha"]).any():
    raise ValueError("Esperaba 18 fechas y una fila por celda y fecha.")

validas = datos.loc[
    datos["fraccion_valida"].ge(0.70)
    & datos["temperatura_mediana_c"].notna()
].copy()

fechas_por_celda = validas.groupby("id_celda")["fecha"].nunique()
ids_comunes = fechas_por_celda.index[fechas_por_celda.eq(n_fechas)]
comparables = validas.loc[validas["id_celda"].isin(ids_comunes)].copy()

if comparables.empty:
    raise ValueError("No hay celdas con cobertura suficiente en las 18 fechas.")

p90_por_fecha = (
    comparables.groupby("fecha")["temperatura_mediana_c"].quantile(0.90)
)
comparables["p90_fecha_c"] = comparables["fecha"].map(p90_por_fecha)
comparables["caliente"] = comparables["temperatura_mediana_c"].ge(
    comparables["p90_fecha_c"]
)

por_anio = (
    comparables.groupby(["id_celda", "col_celda", "fila_celda", "anio"], as_index=False)
    .agg(
        fechas_validas=("fecha", "nunique"),
        fechas_calientes=("caliente", "sum"),
    )
)
por_anio["fraccion_anual"] = por_anio["fechas_calientes"] / por_anio["fechas_validas"]

recurrencia_multianual = (
    por_anio.groupby(["id_celda", "col_celda", "fila_celda"], as_index=False)
    .agg(
        recurrencia_media_anual=("fraccion_anual", "mean"),
        fechas_calientes=("fechas_calientes", "sum"),
        fechas_validas=("fechas_validas", "sum"),
        anios_con_al_menos_una_fecha_caliente=("fechas_calientes", lambda x: x.gt(0).sum()),
    )
)

print("Celdas comparables en las 18 fechas:", len(recurrencia_multianual))
print(
    "Celdas calientes al menos una vez en cada año:",
    recurrencia_multianual["anios_con_al_menos_una_fecha_caliente"].eq(4).sum(),
)
print(
    "Celdas calientes en las 18 fechas:",
    recurrencia_multianual["fechas_calientes"].eq(n_fechas).sum(),
)

geometrias = [
    box(col * 250, fila * 250, (col + 1) * 250, (fila + 1) * 250)
    for col, fila in zip(
        recurrencia_multianual["col_celda"], recurrencia_multianual["fila_celda"]
    )
]
mapa = gpd.GeoDataFrame(recurrencia_multianual, geometry=geometrias, crs="EPSG:32616")
mapa["recurrencia_pct"] = 100 * mapa["recurrencia_media_anual"]

fig, ax = plt.subplots(figsize=(9, 8))
managua.to_crs(mapa.crs).plot(
    ax=ax, color="#e5e5e5", edgecolor="#666666", linewidth=0.8
)
mapa.plot(
    ax=ax,
    column="recurrencia_pct",
    cmap="YlOrRd",
    vmin=0,
    vmax=100,
    legend=True,
    legend_kwds={"label": "Promedio anual de fechas en el P90 o superior (%)"},
)
ax.set_title("Managua · Recurrencia de calor superficial\n18 escenas Landsat de 2023 a 2026")
ax.set_axis_off()

salida = Path("data/celdas_por_escena_2023_2026")
recurrencia_multianual.to_csv(salida / "recurrencia_multianual.csv", index=False)
por_anio.to_csv(salida / "recurrencia_por_anio.csv", index=False)
fig.savefig(salida / "mapa_recurrencia_multianual.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd

datos = panel_multianual.copy()

# Una celda terrestre puede tener solo parte de su superficie en tierra.
# Por eso usamos el número de píxeles terrestres como aproximación de área.
tierra_por_celda = datos.groupby("id_celda")["n_pixeles_tierra"].max()

validas = datos.loc[
    datos["fraccion_valida"].ge(0.70)
    & datos["temperatura_mediana_c"].notna()
]

soporte = validas.groupby("id_celda").agg(
    fechas_validas=("fecha", "nunique"),
    anios_validos=("anio", "nunique"),
)

filas = []

for minimo_fechas in (18, 15, 12):
    cumple = (
        soporte["fechas_validas"].ge(minimo_fechas)
        & soporte["anios_validos"].eq(4)
    )
    ids = soporte.index[cumple]

    filas.append({
        "minimo_fechas": minimo_fechas,
        "celdas_incluidas": len(ids),
        "pct_celdas_terrestres": 100 * len(ids) / len(tierra_por_celda),
        "pct_area_terrestre_aprox": (
            100 * tierra_por_celda.loc[ids].sum() / tierra_por_celda.sum()
        ),
    })

auditoria_cobertura = pd.DataFrame(filas)
display(auditoria_cobertura.round(1))

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

# Reutiliza "validas" y "soporte" de la auditoría de cobertura.
ids_15 = soporte.index[
    soporte["fechas_validas"].ge(15)
    & soporte["anios_validos"].eq(4)
]

datos_15 = validas.loc[validas["id_celda"].isin(ids_15)].copy()

p90_15 = datos_15.groupby("fecha")["temperatura_mediana_c"].quantile(0.90)

datos_15["p90_fecha_c"] = datos_15["fecha"].map(p90_15)
datos_15["caliente"] = datos_15["temperatura_mediana_c"].ge(datos_15["p90_fecha_c"])

anual_15 = (
    datos_15.groupby(["id_celda", "col_celda", "fila_celda", "anio"], as_index=False)
    .agg(
        fechas_validas=("fecha", "nunique"),
        fechas_calientes=("caliente", "sum"),
    )
)
anual_15["fraccion_anual"] = (
    anual_15["fechas_calientes"] / anual_15["fechas_validas"]
)

recurrencia_15 = (
    anual_15.groupby(["id_celda", "col_celda", "fila_celda"], as_index=False)
    .agg(
        recurrencia_media_anual=("fraccion_anual", "mean"),
        fechas_validas=("fechas_validas", "sum"),
        fechas_calientes=("fechas_calientes", "sum"),
        anios_validos=("anio", "nunique"),
    )
)
recurrencia_15["recurrencia_pct"] = 100 * recurrencia_15["recurrencia_media_anual"]

# Comparación con el mapa estricto sobre sus celdas compartidas.
comparacion = recurrencia_multianual[
    ["id_celda", "recurrencia_media_anual"]
].merge(
    recurrencia_15[["id_celda", "recurrencia_media_anual"]],
    on="id_celda",
    suffixes=("_18", "_15"),
    validate="one_to_one",
)

diferencia_pp = 100 * (
    comparacion["recurrencia_media_anual_15"]
    - comparacion["recurrencia_media_anual_18"]
)

print("Celdas del mapa principal:", len(recurrencia_15))
print("Celdas compartidas con el mapa estricto:", len(comparacion))
print(f"Cambio absoluto mediano: {diferencia_pp.abs().median():.1f} puntos porcentuales")
print(f"El 90 % cambió hasta {diferencia_pp.abs().quantile(0.90):.1f} puntos")

geometrias = [
    box(col * 250, fila * 250, (col + 1) * 250, (fila + 1) * 250)
    for col, fila in zip(recurrencia_15["col_celda"], recurrencia_15["fila_celda"])
]
mapa_15 = gpd.GeoDataFrame(recurrencia_15, geometry=geometrias, crs="EPSG:32616")

fig, ax = plt.subplots(figsize=(9, 8))
managua.to_crs(mapa_15.crs).plot(
    ax=ax, color="#e5e5e5", edgecolor="#666666", linewidth=0.8
)
mapa_15.plot(
    ax=ax,
    column="recurrencia_pct",
    cmap="YlOrRd",
    vmin=0,
    vmax=100,
    legend=True,
    legend_kwds={"label": "Promedio anual de fechas en el P90 o superior (%)"},
)

ax.set_title("Managua · Recurrencia de calor superficial\nAl menos 15 de 18 fechas · 2023–2026")
ax.set_axis_off()

carpeta = Path("data/celdas_por_escena_2023_2026")
recurrencia_15.to_csv(carpeta / "recurrencia_multianual_15_fechas.csv", index=False)
fig.savefig(carpeta / "mapa_recurrencia_15_fechas.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path

import pandas as pd

carpeta = Path("data/predictores")
carpeta.mkdir(parents=True, exist_ok=True)

cuadricula = (
    panel_multianual[["id_celda", "col_celda", "fila_celda"]]
    .drop_duplicates()
    .sort_values("id_celda")
    .reset_index(drop=True)
)

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
)
tierra_wc = worldcover.neq(80)

arboles = worldcover.eq(10).rename("fraccion_arboles")
construido = worldcover.eq(50).rename("fraccion_construida")

elevacion = (
    ee.ImageCollection("COPERNICUS/DEM/GLO30_2024_1")
    .filterBounds(area_managua)
    .select("DEM")
    .mosaic()
    .rename("elevacion_media_m")
)

capas = (
    arboles.addBands(construido)
    .addBands(elevacion)
    .updateMask(tierra_wc)
    .clip(area_managua)
)

columnas = [
    "id_celda",
    "fraccion_arboles",
    "fraccion_construida",
    "elevacion_media_m",
]

tamano_bloque = 250
archivos = []

for inicio in range(0, len(cuadricula), tamano_bloque):
    bloque = cuadricula.iloc[inicio:inicio + tamano_bloque]
    archivo = carpeta / f"worldcover_dem_{inicio:05d}.csv"
    archivos.append(archivo)

    if archivo.exists():
        continue

    celdas_ee = []
    for celda in bloque.itertuples(index=False):
        x = int(celda.col_celda) * 250
        y = int(celda.fila_celda) * 250
        rectangulo = ee.Geometry.Rectangle(
            [x, y, x + 250, y + 250],
            proj="EPSG:32616",
            geodesic=False,
        )
        celdas_ee.append(ee.Feature(rectangulo, {"id_celda": celda.id_celda}))

    resumen = capas.reduceRegions(
        collection=ee.FeatureCollection(celdas_ee),
        reducer=ee.Reducer.mean(),
        crs="EPSG:32616",
        scale=10,
        tileScale=4,
    )

    # Quitamos las geometrías para devolver solo la tabla.
    registros = resumen.map(
        lambda feature: ee.Feature(None, feature.toDictionary(columnas))
    ).getInfo()["features"]

    tabla_bloque = pd.DataFrame(
        [registro["properties"] for registro in registros],
        columns=columnas,
    )

    if len(tabla_bloque) != len(bloque):
        raise ValueError(f"El bloque {inicio} devolvió un número inesperado de filas.")

    temporal = archivo.with_suffix(".part")
    tabla_bloque.to_csv(temporal, index=False)
    temporal.replace(archivo)

    print(f"Bloque {inicio:05d} | {len(tabla_bloque)} celdas guardadas")

predictores = pd.concat(
    [pd.read_csv(archivo, dtype={"id_celda": "string"}) for archivo in archivos],
    ignore_index=True,
)

predictores.to_csv(carpeta / "predictores_managua_250m.csv", index=False)

print("\nCeldas con registro:", len(predictores))
print("Valores faltantes por predictor:")
print(predictores[columnas[1:]].isna().sum())
display(predictores.head())

In [ ]:
panel_modelo = panel_multianual.merge(
    predictores,
    on="id_celda",
    how="left",
    validate="many_to_one",
)

panel_modelo = panel_modelo.loc[
    panel_modelo["fraccion_valida"].ge(0.70)
    & panel_modelo["temperatura_mediana_c"].notna()
].copy()

entrenamiento = panel_modelo.loc[panel_modelo["anio"].lt(2026)].copy()
prueba_2026 = panel_modelo.loc[panel_modelo["anio"].eq(2026)].copy()

for nombre, tabla in [
    ("Entrenamiento 2023–2025", entrenamiento),
    ("Prueba retrospectiva 2026", prueba_2026),
]:
    print(
        f"{nombre}: {len(tabla):,} filas | "
        f"{tabla['fecha'].nunique()} fechas | "
        f"{tabla['id_celda'].nunique():,} celdas"
    )

panel_modelo.to_csv(
    carpeta / "managua_panel_con_predictores.csv",
    index=False,
)

Fase de modelacion 

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

datos = panel_modelo.copy()
datos["fecha"] = pd.to_datetime(datos["fecha"])
datos = datos.sort_values(["fecha", "id_celda"]).reset_index(drop=True)

if datos.duplicated(["id_celda", "fecha"]).any():
    raise ValueError("Hay más de una fila para alguna celda y fecha.")

p90_observado = (
    datos.groupby("fecha")["temperatura_mediana_c"]
    .transform("quantile", q=0.90)
)
datos["caliente"] = datos["temperatura_mediana_c"].ge(p90_observado).astype(int)

historico = datos.loc[datos["anio"].lt(2026)].copy()
prueba = datos.loc[datos["anio"].eq(2026)].copy()

# Historial de cada celda disponible justo antes de cada fecha de entrenamiento.
por_celda = historico.groupby("id_celda", sort=False)
historico["n_previas"] = por_celda.cumcount()
historico["calientes_previas"] = por_celda["caliente"].cumsum() - historico["caliente"]

# Historial fijo al cierre de 2025 para todas las fechas de prueba.
historial_final = (
    historico.groupby("id_celda", as_index=False)
    .agg(
        n_previas=("caliente", "size"),
        calientes_previas=("caliente", "sum"),
    )
)
prueba = prueba.merge(
    historial_final, on="id_celda", how="left", validate="many_to_one"
)
prueba[["n_previas", "calientes_previas"]] = (
    prueba[["n_previas", "calientes_previas"]].fillna(0)
)

# Suavizado hacia 10 % con el peso de dos observaciones.
# Una celda sin historial parte de 10 %.
for tabla in (historico, prueba):
    tabla["frecuencia_previa"] = (
        (tabla["calientes_previas"] + 0.20) / (tabla["n_previas"] + 2)
    )

variables_historial = ["frecuencia_previa", "n_previas"]
variables_entorno = variables_historial + [
    "fraccion_arboles",
    "fraccion_construida",
    "elevacion_media_m",
]

for nombre, variables in [
    ("historial", variables_historial),
    ("historial_entorno", variables_entorno),
]:
    modelo = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000),
    )
    modelo.fit(historico[variables], historico["caliente"])
    prueba[f"prob_{nombre}"] = modelo.predict_proba(prueba[variables])[:, 1]

filas = []

for fecha, escena in prueba.groupby("fecha", sort=True):
    y = escena["caliente"]
    fila = {"fecha": fecha.date(), "celdas": len(escena)}

    for nombre in ("historial", "historial_entorno"):
        prob = escena[f"prob_{nombre}"]
        fila[f"AP_{nombre}"] = average_precision_score(y, prob)
        fila[f"Brier_{nombre}"] = brier_score_loss(y, prob)

    fila["mejora_AP"] = fila["AP_historial_entorno"] - fila["AP_historial"]
    fila["mejora_Brier"] = fila["Brier_historial"] - fila["Brier_historial_entorno"]
    filas.append(fila)

metricas_por_fecha = pd.DataFrame(filas)

display(metricas_por_fecha.round(4))
display(metricas_por_fecha.drop(columns=["fecha", "celdas"]).mean().round(4))

Ablacion

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

base = ["frecuencia_previa", "n_previas"]

variantes = {
    "Solo historial": base,
    "Historial + árboles": base + ["fraccion_arboles"],
    "Historial + superficie construida": base + ["fraccion_construida"],
    "Historial + elevación": base + ["elevacion_media_m"],
    "Historial + entorno completo": base + [
        "fraccion_arboles",
        "fraccion_construida",
        "elevacion_media_m",
    ],
}

predicciones_ablacion = prueba[["fecha", "id_celda", "caliente"]].copy()

for nombre, variables in variantes.items():
    modelo = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000),
    )
    modelo.fit(historico[variables], historico["caliente"])
    predicciones_ablacion[nombre] = modelo.predict_proba(prueba[variables])[:, 1]

filas = []

for fecha, escena in predicciones_ablacion.groupby("fecha", sort=True):
    y = escena["caliente"]
    ap_base = average_precision_score(y, escena["Solo historial"])
    brier_base = brier_score_loss(y, escena["Solo historial"])

    for nombre in variantes:
        ap = average_precision_score(y, escena[nombre])
        brier = brier_score_loss(y, escena[nombre])

        filas.append({
            "fecha": fecha.date(),
            "modelo": nombre,
            "AP": ap,
            "Brier": brier,
            "mejora_AP": ap - ap_base,
            "mejora_Brier": brier_base - brier,
        })

ablacion_por_fecha = pd.DataFrame(filas)

resumen_ablacion = (
    ablacion_por_fecha.groupby("modelo", sort=False)
    .agg(
        AP_media=("AP", "mean"),
        Brier_medio=("Brier", "mean"),
        mejora_AP_media=("mejora_AP", "mean"),
        mejora_Brier_media=("mejora_Brier", "mean"),
        fechas_mejora_AP=("mejora_AP", lambda valores: valores.gt(0).sum()),
        fechas_mejora_Brier=("mejora_Brier", lambda valores: valores.gt(0).sum()),
    )
    .sort_values("AP_media", ascending=False)
)

display(resumen_ablacion.round(4))

print("\nMejora de AP por fecha frente a solo historial")
display(
    ablacion_por_fecha.pivot(
        index="fecha",
        columns="modelo",
        values="mejora_AP",
    )
    .drop(columns="Solo historial")
    .round(4)
)

In [ ]:
entorno = [
    "fraccion_arboles",
    "fraccion_construida",
    "elevacion_media_m",
]

referencia = (
    ablacion_por_fecha.loc[
        ablacion_por_fecha["modelo"].eq("Historial + entorno completo"),
        ["fecha", "AP", "Brier"],
    ]
    .set_index("fecha")
)

resultados = []

for variable_retirada in entorno:
    variables = base + [
        variable for variable in entorno if variable != variable_retirada
    ]

    modelo = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000),
    )
    modelo.fit(historico[variables], historico["caliente"])

    evaluacion = prueba[["fecha", "caliente"]].copy()
    evaluacion["probabilidad"] = modelo.predict_proba(prueba[variables])[:, 1]

    for fecha, escena in evaluacion.groupby("fecha", sort=True):
        fecha = fecha.date()
        y = escena["caliente"]
        prob = escena["probabilidad"]

        resultados.append({
            "fecha": fecha,
            "variable_retirada": variable_retirada,
            "perdida_AP_al_quitar": (
                referencia.at[fecha, "AP"] - average_precision_score(y, prob)
            ),
            "aumento_Brier_al_quitar": (
                brier_score_loss(y, prob) - referencia.at[fecha, "Brier"]
            ),
        })

ablacion_inversa = pd.DataFrame(resultados)

display(
    ablacion_inversa.groupby("variable_retirada").agg(
        perdida_AP_media=("perdida_AP_al_quitar", "mean"),
        aumento_Brier_medio=("aumento_Brier_al_quitar", "mean"),
        fechas_con_perdida_AP=("perdida_AP_al_quitar", lambda x: x.gt(0).sum()),
        fechas_con_aumento_Brier=("aumento_Brier_al_quitar", lambda x: x.gt(0).sum()),
    ).round(4)
)

In [ ]:
import pandas as pd
from ngboost import NGBRegressor
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

objetivo = "temperatura_mediana_c"

datos = panel_modelo.copy()
datos["fecha"] = pd.to_datetime(datos["fecha"])
datos = datos.sort_values(["fecha", "id_celda"]).reset_index(drop=True)

if datos.duplicated(["id_celda", "fecha"]).any():
    raise ValueError("Hay más de una fila para alguna celda y fecha.")

datos["dia_del_anio"] = datos["fecha"].dt.dayofyear
train_temp = datos.loc[datos["anio"].lt(2026)].copy()
test_temp = datos.loc[datos["anio"].eq(2026)].copy()

# Para cada fila de entrenamiento, usamos únicamente escenas anteriores.
por_celda = train_temp.groupby("id_celda", sort=False)
train_temp["n_previas"] = por_celda.cumcount()
suma_previa = por_celda[objetivo].cumsum() - train_temp[objetivo]
train_temp["media_previa_c"] = suma_previa.div(
    train_temp["n_previas"].where(train_temp["n_previas"].gt(0))
)

# El historial de prueba queda congelado al cierre de 2025.
historial_temp = train_temp.groupby("id_celda", as_index=False).agg(
    media_previa_c=(objetivo, "mean"),
    n_previas=(objetivo, "size"),
)
test_temp = test_temp.merge(
    historial_temp, on="id_celda", how="left", validate="many_to_one"
)
test_temp["n_previas"] = test_temp["n_previas"].fillna(0)

variables = [
    "media_previa_c",
    "n_previas",
    "dia_del_anio",
    "fraccion_arboles",
    "fraccion_construida",
    "elevacion_media_m",
]

x_train = train_temp[variables]
y_train = train_temp[objetivo]
x_test = test_temp[variables]

# Referencia histórica. Si la celda nunca se observó, usamos la media de entrenamiento.
test_temp["pred_historial_c"] = test_temp["media_previa_c"].fillna(y_train.mean())

ridge = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    Ridge(alpha=1.0),
)
ridge.fit(x_train, y_train)
test_temp["pred_ridge_c"] = ridge.predict(x_test)

imputador = SimpleImputer(strategy="median", add_indicator=True)
x_train_ngb = imputador.fit_transform(x_train)
x_test_ngb = imputador.transform(x_test)

ngb = NGBRegressor(
    Dist=Normal,
    Score=LogScore,
    Base=DecisionTreeRegressor(max_depth=2, min_samples_leaf=150),
    n_estimators=300,
    learning_rate=0.03,
    random_state=42,
    verbose=False,
)
ngb.fit(x_train_ngb, y_train.to_numpy())

distribucion = ngb.pred_dist(x_test_ngb)
test_temp["pred_ngboost_c"] = distribucion.params["loc"]
test_temp["desvio_ngboost_c"] = distribucion.params["scale"]
test_temp["limite_inferior_90_c"] = distribucion.ppf(0.05)
test_temp["limite_superior_90_c"] = distribucion.ppf(0.95)

# CRPS de la distribución normal. Menor es mejor.
z = (
    (test_temp[objetivo] - test_temp["pred_ngboost_c"])
    / test_temp["desvio_ngboost_c"]
)
test_temp["crps_ngboost_c"] = test_temp["desvio_ngboost_c"] * (
    z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / (3.141592653589793 ** 0.5)
)

filas = []
for fecha, escena in test_temp.groupby("fecha", sort=True):
    y = escena[objetivo]
    fila = {"fecha": fecha.date(), "celdas": len(escena)}

    for modelo in ("historial", "ridge", "ngboost"):
        pred = escena[f"pred_{modelo}_c"]
        fila[f"MAE_{modelo}_c"] = mean_absolute_error(y, pred)
        fila[f"RMSE_{modelo}_c"] = mean_squared_error(y, pred) ** 0.5

    inferior = escena["limite_inferior_90_c"]
    superior = escena["limite_superior_90_c"]
    fila["cobertura_90_pct"] = 100 * (y.ge(inferior) & y.le(superior)).mean()
    fila["ancho_intervalo_90_c"] = (superior - inferior).mean()
    fila["CRPS_ngboost_c"] = escena["crps_ngboost_c"].mean()
    filas.append(fila)

metricas_temp = pd.DataFrame(filas)

display(metricas_temp.round(3))
display(metricas_temp.drop(columns=["fecha", "celdas"]).mean().round(3))

In [ ]:
import pandas as pd

filas = []

for fecha, escena in test_temp.groupby("fecha", sort=True):
    observado = escena["temperatura_mediana_c"]
    inferior = escena["limite_inferior_90_c"]
    superior = escena["limite_superior_90_c"]

    for modelo in ("ridge", "ngboost"):
        error = escena[f"pred_{modelo}_c"] - observado
        sesgo = error.median()

        fila = {
            "fecha": pd.Timestamp(fecha).date(),
            "modelo": modelo,
            "sesgo_mediano_c": sesgo,
            "MAE_c": error.abs().mean(),
            "MAE_sin_desplazamiento_c": (error - sesgo).abs().mean(),
        }

        if modelo == "ngboost":
            fila["pct_observado_bajo_intervalo"] = 100 * observado.lt(inferior).mean()
            fila["pct_observado_sobre_intervalo"] = 100 * observado.gt(superior).mean()

        filas.append(fila)

diagnostico_temp = pd.DataFrame(filas)
display(diagnostico_temp.round(2))

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

escenas = (
    pd.concat([train_temp, test_temp], ignore_index=True)
    [["fecha", "id_escena"]]
    .drop_duplicates()
    .sort_values("fecha")
)
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY").select("temperature_2m")

registros = []

for fila in escenas.itertuples(index=False):
    paso = pd.Timestamp(ee.Image(fila.id_escena).get("system:time_start").getInfo(),
                        unit="ms", tz="UTC")
    hora = paso.floor("h")
    inicio = ee.Date(int(hora.timestamp() * 1000))
    hora_era5 = era5.filterDate(inicio, inicio.advance(1, "hour"))

    if hora_era5.size().getInfo() != 1:
        raise ValueError(f"Falta una imagen ERA5-Land para {hora}.")

    kelvin = ee.Image(hora_era5.first()).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area_managua,
        scale=11132,
        maxPixels=100_000,
    ).get("temperature_2m").getInfo()

    if kelvin is None:
        raise ValueError(f"ERA5-Land no devolvió temperatura para {hora}.")

    registros.append({
        "id_escena": fila.id_escena,
        "fecha": fila.fecha,
        "hora_era5_utc": hora,
        "temp_aire_c": kelvin - 273.15,
    })

meteo = pd.DataFrame(registros)
carpeta = Path("data/predictores")
carpeta.mkdir(parents=True, exist_ok=True)
meteo.to_csv(carpeta / "managua_era5_land_por_escena.csv", index=False)

train_meteo = train_temp.merge(
    meteo[["id_escena", "temp_aire_c"]], on="id_escena", validate="many_to_one"
)
test_meteo = test_temp.merge(
    meteo[["id_escena", "temp_aire_c"]], on="id_escena", validate="many_to_one"
)

if len(train_meteo) != len(train_temp) or len(test_meteo) != len(test_temp):
    raise ValueError("La unión meteorológica cambió el conjunto de evaluación.")

variables_meteo = variables + ["temp_aire_c"]
ridge_meteo = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    Ridge(alpha=1.0),
)
ridge_meteo.fit(train_meteo[variables_meteo], train_meteo[objetivo])
test_meteo["pred_ridge_meteo_c"] = ridge_meteo.predict(test_meteo[variables_meteo])

filas = []
for fecha, escena in test_meteo.groupby("fecha", sort=True):
    y = escena[objetivo]
    base = escena["pred_ridge_c"]
    nueva = escena["pred_ridge_meteo_c"]

    mae_base = mean_absolute_error(y, base)
    mae_meteo = mean_absolute_error(y, nueva)
    filas.append({
        "fecha": fecha,
        "temp_aire_c": escena["temp_aire_c"].mean(),
        "MAE_ridge_c": mae_base,
        "MAE_ridge_meteo_c": mae_meteo,
        "mejora_MAE_c": mae_base - mae_meteo,
        "RMSE_ridge_meteo_c": mean_squared_error(y, nueva) ** 0.5,
        "sesgo_ridge_c": (base - y).median(),
        "sesgo_ridge_meteo_c": (nueva - y).median(),
    })

metricas_meteo = pd.DataFrame(filas)
display(metricas_meteo.round(3))
print("Mejora media de MAE (°C):", round(metricas_meteo["mejora_MAE_c"].mean(), 3))

In [ ]:
from pathlib import Path

import certifi
import ee

certificado = Path(certifi.where())
print("Certificado encontrado:", certificado.is_file())

if not certificado.is_file():
    raise FileNotFoundError(f"Falta el certificado en {certificado}")

ee.Initialize(project="pm25-nicaragua-earth-engine")
print("Conexión con Earth Engine:", ee.Number(1).getInfo())

In [ ]:
from pathlib import Path

import pandas as pd
import requests

ruta_panel = Path("data/predictores/managua_panel_con_predictores.csv")
panel_modelo = pd.read_csv(ruta_panel, parse_dates=["fecha"])
print(f"Panel cargado: {len(panel_modelo):,} filas")

In [ ]:
necesarios = ["train_temp", "test_temp", "area_managua", "variables", "objetivo"]
faltan = [nombre for nombre in necesarios if nombre not in globals()]
print("Faltan:", faltan)

In [ ]:
from pathlib import Path

import ee
import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import Point, mapping
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Recuperar el panel guardado y separar entrenamiento y prueba.
ruta_panel = Path("data/predictores/managua_panel_con_predictores.csv").resolve()

if not ruta_panel.is_file():
    raise FileNotFoundError(f"No encontré el CSV en {ruta_panel}")

objetivo = "temperatura_mediana_c"
datos = pd.read_csv(ruta_panel, parse_dates=["fecha"])
datos = datos.sort_values(["fecha", "id_celda"]).reset_index(drop=True)
datos["dia_del_anio"] = datos["fecha"].dt.dayofyear

train_temp = datos.loc[datos["anio"].between(2023, 2025)].copy()
test_temp = datos.loc[datos["anio"].eq(2026)].copy()

# 2. Reconstruir el historial de cada celda sin usar fechas futuras.
por_celda = train_temp.groupby("id_celda", sort=False)
train_temp["n_previas"] = por_celda.cumcount()
suma_previa = por_celda[objetivo].cumsum() - train_temp[objetivo]
train_temp["media_previa_c"] = suma_previa.div(
    train_temp["n_previas"].where(train_temp["n_previas"].gt(0))
)

historial_temp = train_temp.groupby("id_celda", as_index=False).agg(
    media_previa_c=(objetivo, "mean"),
    n_previas=(objetivo, "size"),
)
test_temp = test_temp.merge(
    historial_temp, on="id_celda", how="left", validate="many_to_one"
)
test_temp["n_previas"] = test_temp["n_previas"].fillna(0)

# 3. Reproducir el Ridge de referencia.
variables = [
    "media_previa_c",
    "n_previas",
    "dia_del_anio",
    "fraccion_arboles",
    "fraccion_construida",
    "elevacion_media_m",
]

ridge = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    Ridge(alpha=1.0),
)
ridge.fit(train_temp[variables], train_temp[objetivo])
test_temp["pred_ridge_c"] = ridge.predict(test_temp[variables])

# 4. Recuperar el mismo polígono municipal usado para Landsat.
url_limites = "https://www.geoboundaries.org/api/current/gbOpen/NIC/ADM2/"
respuesta = requests.get(url_limites, timeout=60)
respuesta.raise_for_status()

municipios = gpd.read_file(respuesta.json()["gjDownloadURL"]).to_crs("EPSG:4326")
punto_centro = Point(-86.2730, 12.1364)
managua = municipios.loc[municipios.geometry.covers(punto_centro)].copy()

if len(managua) != 1:
    raise ValueError(f"El punto de referencia coincidió con {len(managua)} municipios.")

area_managua = ee.Geometry(mapping(managua.geometry.iloc[0]))

print(f"Entrenamiento: {len(train_temp):,} filas | {train_temp['fecha'].nunique()} fechas")
print(f"Prueba: {len(test_temp):,} filas | {test_temp['fecha'].nunique()} fechas")
print("Listo para ejecutar la celda de ERA5-Land.")

In [ ]:
from pathlib import Path

import ee
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

necesarios = ["train_temp", "test_temp", "area_managua", "variables", "objetivo"]
faltan = [nombre for nombre in necesarios if nombre not in globals()]

if faltan:
    raise RuntimeError(f"Faltan estos objetos del notebook: {', '.join(faltan)}")

escenas = (
    pd.concat([train_temp[["id_escena"]], test_temp[["id_escena"]]])
    ["id_escena"]
    .drop_duplicates()
)
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY").select("temperature_2m")

# Guardamos cada escena para poder retomar si se corta la conexión.
carpeta_meteo = Path("data/predictores/era5_por_escena")
carpeta_meteo.mkdir(parents=True, exist_ok=True)
tablas_meteo = []

for id_escena in escenas:
    archivo = carpeta_meteo / f"{id_escena.rsplit('/', 1)[-1]}.csv"

    if archivo.exists():
        tablas_meteo.append(pd.read_csv(archivo))
        continue

    tiempo_ms = ee.Image(id_escena).get("system:time_start").getInfo()
    paso = pd.to_datetime(tiempo_ms, unit="ms", utc=True)
    hora = paso.floor("h")
    inicio = ee.Date(int(hora.timestamp() * 1000))
    coleccion_hora = era5.filterDate(inicio, inicio.advance(1, "hour"))

    if coleccion_hora.size().getInfo() != 1:
        raise ValueError(f"ERA5-Land no devolvió una imagen para {hora}.")

    kelvin = ee.Image(coleccion_hora.first()).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area_managua,
        scale=11132,
        maxPixels=100_000,
    ).get("temperature_2m").getInfo()

    if kelvin is None:
        raise ValueError(f"Falta la temperatura de ERA5-Land para {hora}.")

    registro = pd.DataFrame([{
        "id_escena": id_escena,
        "hora_era5_utc": hora.isoformat(),
        "temp_aire_c": kelvin - 273.15,
    }])
    temporal = archivo.with_suffix(".part")
    registro.to_csv(temporal, index=False)
    temporal.replace(archivo)
    tablas_meteo.append(registro)
    print(f"{hora:%Y-%m-%d %H:%M} UTC | {kelvin - 273.15:.1f} °C")

meteo = pd.concat(tablas_meteo, ignore_index=True)
columnas_meteo = ["id_escena", "temp_aire_c"]

train_meteo = train_temp.merge(
    meteo[columnas_meteo], on="id_escena", how="left", validate="many_to_one"
)
test_meteo = test_temp.merge(
    meteo[columnas_meteo], on="id_escena", how="left", validate="many_to_one"
)

if train_meteo["temp_aire_c"].isna().any() or test_meteo["temp_aire_c"].isna().any():
    raise ValueError("Hay observaciones sin temperatura del aire.")

variables_meteo = list(variables) + ["temp_aire_c"]
ridge_meteo = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    Ridge(alpha=1.0),
)
ridge_meteo.fit(train_meteo[variables_meteo], train_meteo[objetivo])
test_meteo["pred_ridge_meteo_c"] = ridge_meteo.predict(test_meteo[variables_meteo])

filas = []

for fecha, escena in test_meteo.groupby("fecha", sort=True):
    observado = escena[objetivo]
    pred_base = escena["pred_ridge_c"]
    pred_meteo = escena["pred_ridge_meteo_c"]

    mae_base = mean_absolute_error(observado, pred_base)
    mae_meteo = mean_absolute_error(observado, pred_meteo)

    filas.append({
        "fecha": pd.Timestamp(fecha).date(),
        "temp_aire_c": escena["temp_aire_c"].mean(),
        "MAE_ridge_c": mae_base,
        "MAE_ridge_meteo_c": mae_meteo,
        "mejora_MAE_c": mae_base - mae_meteo,
        "sesgo_ridge_c": (pred_base - observado).median(),
        "sesgo_ridge_meteo_c": (pred_meteo - observado).median(),
    })

metricas_meteo = pd.DataFrame(filas)
display(metricas_meteo.round(3))
print(f"Mejora media de MAE: {metricas_meteo['mejora_MAE_c'].mean():.3f} °C")

In [ ]:
from pathlib import Path

import ee
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

carpeta_solar = Path("data/predictores/era5_radiacion_desde_06")
carpeta_solar.mkdir(parents=True, exist_ok=True)

banda_solar = "surface_solar_radiation_downwards_hourly"
era5_solar = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY").select(banda_solar)

escenas = train_temp["id_escena"].drop_duplicates()
registros_solares = []

for id_escena in escenas:
    archivo = carpeta_solar / f"{id_escena.rsplit('/', 1)[-1]}.csv"

    if archivo.exists():
        registros_solares.append(pd.read_csv(archivo))
        continue

    tiempo_ms = ee.Image(id_escena).get("system:time_start").getInfo()
    paso_utc = pd.to_datetime(tiempo_ms, unit="ms", utc=True)
    paso_local = paso_utc.tz_convert("America/Managua")

    inicio_local = paso_local.normalize() + pd.Timedelta(hours=6)
    fin_local = paso_local.floor("h")
    n_horas = int((fin_local - inicio_local) / pd.Timedelta(hours=1))

    if n_horas < 1:
        raise ValueError(f"{id_escena}: no hay horas completas en la ventana solar.")

    # Cada valor horario se etiqueta con la hora en que termina el intervalo.
    # Para sumar de 06:00 a 10:00, usamos los valores de 07:00 a 10:00.
    primera_hora = inicio_local + pd.Timedelta(hours=1)
    fin_exclusivo = fin_local + pd.Timedelta(hours=1)

    horas = era5_solar.filterDate(
        ee.Date(int(primera_hora.timestamp() * 1000)),
        ee.Date(int(fin_exclusivo.timestamp() * 1000)),
    )

    n_disponibles = horas.size().getInfo()
    if n_disponibles != n_horas:
        raise ValueError(
            f"{id_escena}: esperaba {n_horas} horas y encontré {n_disponibles}."
        )

    energia = horas.sum().reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area_managua,
        scale=11132,
        maxPixels=100_000,
    ).get(banda_solar).getInfo()

    if energia is None:
        raise ValueError(f"ERA5-Land no devolvió radiación para {id_escena}.")

    registro = pd.DataFrame([{
        "id_escena": id_escena,
        "radiacion_solar_mj_m2": energia / 1_000_000,
    }])
    temporal = archivo.with_suffix(".part")
    registro.to_csv(temporal, index=False)
    temporal.replace(archivo)
    registros_solares.append(registro)
    print(f"{paso_local:%Y-%m-%d} | {energia / 1_000_000:.2f} MJ/m²")

solar = pd.concat(registros_solares, ignore_index=True)
datos_solar = train_temp.merge(
    solar, on="id_escena", how="left", validate="many_to_one"
)
datos_solar["fecha"] = pd.to_datetime(datos_solar["fecha"])

if datos_solar["radiacion_solar_mj_m2"].isna().any():
    raise ValueError("Hay escenas sin radiación solar.")

fechas = sorted(datos_solar["fecha"].unique())
if len(fechas) != 12:
    raise ValueError(f"Esperaba 12 fechas de 2023–2025. Encontré {len(fechas)}.")

versiones = {
    "ridge": list(variables),
    "ridge_solar": list(variables) + ["radiacion_solar_mj_m2"],
}

filas = []
for fecha in fechas[6:]:
    ajuste = datos_solar.loc[datos_solar["fecha"].lt(fecha)]
    evaluacion = datos_solar.loc[datos_solar["fecha"].eq(fecha)]
    observado = evaluacion[objetivo]

    fila = {
        "fecha": pd.Timestamp(fecha).date(),
        "fechas_entrenamiento": ajuste["fecha"].nunique(),
        "radiacion_mj_m2": evaluacion["radiacion_solar_mj_m2"].mean(),
    }

    for nombre, columnas in versiones.items():
        modelo = make_pipeline(
            SimpleImputer(strategy="median", add_indicator=True),
            StandardScaler(),
            Ridge(alpha=1.0),
        )
        modelo.fit(ajuste[columnas], ajuste[objetivo])
        prediccion = modelo.predict(evaluacion[columnas])

        fila[f"MAE_{nombre}_c"] = mean_absolute_error(observado, prediccion)
        fila[f"sesgo_{nombre}_c"] = (prediccion - observado).median()

    fila["mejora_MAE_c"] = fila["MAE_ridge_c"] - fila["MAE_ridge_solar_c"]
    filas.append(fila)

metricas_solar = pd.DataFrame(filas)
display(metricas_solar.round(3))
print(f"Mejora media de MAE: {metricas_solar['mejora_MAE_c'].mean():.3f} °C")
print(f"Fechas con menor MAE: {metricas_solar['mejora_MAE_c'].gt(0).sum()} de 6")

In [ ]:
radiacion_por_fecha = (
    datos_solar.groupby("fecha")["radiacion_solar_mj_m2"]
    .mean()
    .sort_index()
)

fecha_revision = pd.Timestamp("2024-03-24")
radiacion_ajuste = radiacion_por_fecha.loc[
    radiacion_por_fecha.index < fecha_revision
]
radiacion_evaluada = radiacion_por_fecha.loc[fecha_revision]

display(radiacion_ajuste.rename("radiacion_mj_m2").round(3).to_frame())
print(f"Rango de ajuste: {radiacion_ajuste.min():.3f} a "
      f"{radiacion_ajuste.max():.3f} MJ/m²")
print(f"24 de marzo: {radiacion_evaluada:.3f} MJ/m²")
print("¿Fuera del rango de ajuste?",
      not radiacion_ajuste.min() <= radiacion_evaluada <= radiacion_ajuste.max())

In [ ]:
resultados_regla = metricas_solar.copy()
resultados_regla["fecha"] = pd.to_datetime(resultados_regla["fecha"])

radiacion_por_fecha = (
    datos_solar.groupby("fecha")["radiacion_solar_mj_m2"]
    .mean()
    .sort_index()
)

# El rango de cada fecha se calcula únicamente con escenas anteriores.
rangos_previos = pd.DataFrame({
    "min_previo": radiacion_por_fecha.cummin().shift(1),
    "max_previo": radiacion_por_fecha.cummax().shift(1),
})

resultados_regla = resultados_regla.join(rangos_previos, on="fecha")

dentro = resultados_regla["radiacion_mj_m2"].between(
    resultados_regla["min_previo"],
    resultados_regla["max_previo"],
    inclusive="both",
)

resultados_regla["modelo_usado"] = dentro.map({
    True: "Ridge + radiación",
    False: "Ridge base",
})
resultados_regla["MAE_regla_c"] = resultados_regla["MAE_ridge_solar_c"].where(
    dentro, resultados_regla["MAE_ridge_c"]
)
resultados_regla["mejora_regla_c"] = (
    resultados_regla["MAE_ridge_c"] - resultados_regla["MAE_regla_c"]
)

display(resultados_regla[[
    "fecha", "radiacion_mj_m2", "min_previo", "max_previo",
    "modelo_usado", "MAE_ridge_c", "MAE_regla_c", "mejora_regla_c",
]].round(3))

print(f"MAE medio con la regla: {resultados_regla['MAE_regla_c'].mean():.3f} °C")
print(f"Mejora media frente a Ridge: {resultados_regla['mejora_regla_c'].mean():.3f} °C")

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

col_solar = "radiacion_solar_mj_m2"
registros_2026 = []

for id_escena in test_temp["id_escena"].drop_duplicates():
    archivo = carpeta_solar / f"{id_escena.rsplit('/', 1)[-1]}.csv"

    if archivo.exists():
        registros_2026.append(pd.read_csv(archivo))
        continue

    tiempo_ms = ee.Image(id_escena).get("system:time_start").getInfo()
    paso = pd.to_datetime(tiempo_ms, unit="ms", utc=True).tz_convert("America/Managua")
    inicio = paso.normalize() + pd.Timedelta(hours=6)
    fin = paso.floor("h")
    n_horas = int((fin - inicio) / pd.Timedelta(hours=1))

    horas = era5_solar.filterDate(
        ee.Date(int((inicio + pd.Timedelta(hours=1)).timestamp() * 1000)),
        ee.Date(int((fin + pd.Timedelta(hours=1)).timestamp() * 1000)),
    )

    if n_horas < 1 or horas.size().getInfo() != n_horas:
        raise ValueError(f"{id_escena}: faltan horas en la ventana de radiación.")

    energia = horas.sum().reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area_managua,
        scale=11132,
        maxPixels=100_000,
    ).get(banda_solar).getInfo()

    if energia is None:
        raise ValueError(f"ERA5-Land no devolvió radiación para {id_escena}.")

    registro = pd.DataFrame([{
        "id_escena": id_escena,
        col_solar: energia / 1_000_000,
    }])
    temporal = archivo.with_suffix(".part")
    registro.to_csv(temporal, index=False)
    temporal.replace(archivo)
    registros_2026.append(registro)

solar_2026 = pd.concat(registros_2026, ignore_index=True)
prueba_solar = test_temp.merge(
    solar_2026, on="id_escena", how="left", validate="many_to_one"
)
if prueba_solar[col_solar].isna().any():
    raise ValueError("Hay observaciones de 2026 sin radiación solar.")

columnas = list(variables) + [col_solar]
ridge_solar = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    Ridge(alpha=1.0),
)
ridge_solar.fit(datos_solar[columnas], datos_solar[objetivo])
prueba_solar["pred_solar_c"] = ridge_solar.predict(prueba_solar[columnas])

min_solar = datos_solar[col_solar].min()
max_solar = datos_solar[col_solar].max()
dentro = prueba_solar[col_solar].between(min_solar, max_solar)

prueba_solar["modelo_usado"] = dentro.map({
    True: "Ridge + radiación",
    False: "Ridge base",
})
prueba_solar["pred_regla_c"] = prueba_solar["pred_solar_c"].where(
    dentro, prueba_solar["pred_ridge_c"]
)

filas = []
for fecha, escena in prueba_solar.groupby("fecha", sort=True):
    observado = escena[objetivo]
    mae_base = mean_absolute_error(observado, escena["pred_ridge_c"])
    mae_regla = mean_absolute_error(observado, escena["pred_regla_c"])
    filas.append({
        "fecha": pd.Timestamp(fecha).date(),
        "radiacion_mj_m2": escena[col_solar].mean(),
        "modelo_usado": escena["modelo_usado"].iloc[0],
        "MAE_ridge_c": mae_base,
        "MAE_solar_c": mean_absolute_error(observado, escena["pred_solar_c"]),
        "MAE_regla_c": mae_regla,
        "mejora_regla_c": mae_base - mae_regla,
    })

metricas_regla_2026 = pd.DataFrame(filas)
print(f"Rango fijo de ajuste: {min_solar:.3f} a {max_solar:.3f} MJ/m²")
display(metricas_regla_2026.round(3))
display(metricas_regla_2026[
    ["MAE_ridge_c", "MAE_solar_c", "MAE_regla_c", "mejora_regla_c"]
].mean().round(3))

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

carpeta = Path("resultados/cierre_managua")
carpeta.mkdir(parents=True, exist_ok=True)

ruta_panel = Path("data/predictores/managua_panel_con_predictores.csv")
datos = pd.read_csv(ruta_panel, parse_dates=["fecha"], dtype={"id_celda": "string"})
objetivo = "temperatura_mediana_c"

datos = (
    datos.loc[
        datos["fecha"].dt.year.between(2023, 2026)
        & datos["fraccion_valida"].ge(0.70)
        & datos[objetivo].notna()
    ]
    .sort_values(["fecha", "id_celda"])
    .reset_index(drop=True)
)

if datos["fecha"].nunique() != 18 or datos.duplicated(["id_celda", "fecha"]).any():
    raise ValueError("Esperaba 18 escenas y una fila por celda y fecha.")

estaticas = ["fraccion_arboles", "fraccion_construida", "elevacion_media_m"]
variables_finales = ["media_previa_c", "n_previas", "dia_del_anio", *estaticas]

por_celda = datos.groupby("id_celda", sort=False)
datos["n_previas"] = por_celda.cumcount()
suma_previa = por_celda[objetivo].cumsum() - datos[objetivo]
datos["media_previa_c"] = suma_previa.div(datos["n_previas"].where(datos["n_previas"].gt(0)))
datos["dia_del_anio"] = datos["fecha"].dt.dayofyear

modelo_final = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    Ridge(alpha=1.0),
)
modelo_final.fit(datos[variables_finales], datos[objetivo])

historial_final = por_celda[objetivo].agg(media_previa_c="mean", n_previas="size")
columnas_celda = ["id_celda", "col_celda", "fila_celda", *estaticas]
celdas_finales = (
    datos[columnas_celda]
    .drop_duplicates()
    .merge(historial_final, left_on="id_celda", right_index=True, validate="one_to_one")
    .sort_values("id_celda")
)

version = "ridge_managua_2023_2026_v1"
paquete = {
    "modelo": modelo_final,
    "variables": variables_finales,
    "version": version,
    "sklearn_version": sklearn.__version__,
    "ultima_escena": datos["fecha"].max().date().isoformat(),
}

joblib.dump(paquete, carpeta / "ridge_managua_v1.joblib")
celdas_finales.to_csv(carpeta / "celdas_e_historial_v1.csv", index=False)

print(f"Entrenamiento final: {len(datos):,} filas, {datos['fecha'].nunique()} escenas")
print(f"Celdas con historial: {len(celdas_finales):,}")
print("Última escena usada:", paquete["ultima_escena"])
print("Guardado en:", carpeta.resolve())

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import box

carpeta = Path("resultados/cierre_managua")
fecha_objetivo = pd.Timestamp("2027-02-15")

paquete = joblib.load(carpeta / "ridge_managua_v1.joblib")
celdas = pd.read_csv(carpeta / "celdas_e_historial_v1.csv", dtype={"id_celda": "string"})

if fecha_objetivo <= pd.Timestamp(paquete["ultima_escena"]):
    raise ValueError("Elegí una fecha posterior a la última escena de entrenamiento.")

celdas["dia_del_anio"] = fecha_objetivo.dayofyear
celdas["temp_superficial_estimada_c"] = paquete["modelo"].predict(
    celdas[paquete["variables"]]
)
celdas["temp_historial_c"] = celdas["media_previa_c"]
celdas["fecha_objetivo"] = fecha_objetivo.date().isoformat()
celdas["version_modelo"] = paquete["version"]
celdas["generada_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")

nombre = f"{paquete['version']}_{fecha_objetivo:%Y-%m-%d}"
ruta_csv = carpeta / f"{nombre}.csv"
ruta_mapa = carpeta / f"{nombre}.png"

columnas_salida = [
    "id_celda", "col_celda", "fila_celda", "fecha_objetivo",
    "temp_superficial_estimada_c", "temp_historial_c", "n_previas",
    "version_modelo", "generada_utc",
]

# Conservá la primera predicción archivada para esa fecha y versión.
if ruta_csv.exists():
    celdas = pd.read_csv(ruta_csv, dtype={"id_celda": "string"})
    print("Se reutilizó la predicción archivada.")
else:
    celdas[columnas_salida].to_csv(ruta_csv, index=False, mode="x")

geometrias = [
    box(col * 250, fila * 250, (col + 1) * 250, (fila + 1) * 250)
    for col, fila in zip(celdas["col_celda"], celdas["fila_celda"])
]
mapa = gpd.GeoDataFrame(celdas, geometry=geometrias, crs="EPSG:32616")

fig, ax = plt.subplots(figsize=(9, 8))
mapa.plot(
    ax=ax,
    column="temp_superficial_estimada_c",
    cmap="inferno",
    legend=True,
    legend_kwds={"label": "Temperatura superficial estimada (°C)", "shrink": 0.75},
)
ax.set_title(f"Managua · Temperatura superficial estimada\n{fecha_objetivo:%d/%m/%Y} · Ridge")
ax.set_axis_off()
fig.savefig(ruta_mapa, dpi=180, bbox_inches="tight")
plt.show()

print("CSV:", ruta_csv.resolve())
print("Mapa:", ruta_mapa.resolve())

In [ ]:
from matplotlib import patheffects

# Nombre, longitud, latitud y desplazamiento de la etiqueta en puntos.
referencias = pd.DataFrame([
    ("El Crucero",             -86.30954, 11.99008,  8,   5),
    ("Teatro Rubén Darío",     -86.27274, 12.15831,  8,   8),
    ("Catedral Metropolitana", -86.26599, 12.13270,  8, -10),
    ("Aeropuerto A. C. Sandino", -86.16977, 12.14174, -8,  8),
    ("7 Sur",                  -86.31320, 12.12419, -8,  -8),
    ("Rotonda de Ticuantepe",   -86.20271, 12.05525,  8,  -8),
    ("Laguna de Tiscapa",       -86.27034, 12.13943, -8,   6),
], columns=["nombre", "lon", "lat", "dx", "dy"])

puntos = gpd.GeoDataFrame(
    referencias,
    geometry=gpd.points_from_xy(referencias["lon"], referencias["lat"]),
    crs="EPSG:4326",
).to_crs("EPSG:32616")

# Combinar el encuadre original con los puntos de referencia.
x0, x1 = ax.get_xlim()
y0, y1 = ax.get_ylim()
px0, py0, px1, py1 = puntos.total_bounds

ax.set_xlim(min(x0, px0) - 1200, max(x1, px1) + 1200)
ax.set_ylim(min(y0, py0) - 900, max(y1, py1) + 900)

ax.scatter(
    puntos.geometry.x, puntos.geometry.y,
    s=32, facecolor="white", edgecolor="#222222",
    linewidth=0.9, zorder=5,
)

for sitio in puntos.itertuples(index=False):
    ax.annotate(
        sitio.nombre,
        xy=(sitio.geometry.x, sitio.geometry.y),
        xytext=(sitio.dx, sitio.dy),
        textcoords="offset points",
        ha="left" if sitio.dx > 0 else "right",
        va="center",
        fontsize=8.5,
        color="#222222",
        path_effects=[patheffects.withStroke(linewidth=2.7, foreground="white")],
        zorder=6,
    )

ax.set_aspect("equal")

# En el mapa de predicción, reemplazá la llamada anterior a fig.savefig por esta.
ruta_mapa_referencias = ruta_mapa.with_name(f"{ruta_mapa.stem}_con_referencias.png")
fig.savefig(ruta_mapa_referencias, dpi=220, bbox_inches="tight")

In [ ]:
from pathlib import Path

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import box

carpeta = Path("resultados/cierre_managua")
paquete = joblib.load(carpeta / "ridge_managua_v1.joblib")
celdas = pd.read_csv(
    carpeta / "celdas_e_historial_v1.csv",
    dtype={"id_celda": "string"},
)

# Fechas de demostración. Reemplazalas por pasos confirmados para evaluar.
fechas_objetivo = pd.to_datetime([
    "2027-01-15", "2027-02-15", "2027-03-15", "2027-04-15"
])

if fechas_objetivo.nunique() != 4 or (
    fechas_objetivo <= pd.Timestamp(paquete["ultima_escena"])
).any():
    raise ValueError("Ingresá cuatro fechas distintas posteriores al entrenamiento.")

# Predicciones del modelo final ajustado con las 18 escenas.
predicciones = []
for fecha in fechas_objetivo:
    entradas = celdas.assign(dia_del_anio=fecha.dayofyear)
    predicciones.append(
        celdas[["id_celda", "col_celda", "fila_celda"]].assign(
            fecha_objetivo=fecha.date().isoformat(),
            temperatura_estimada_c=paquete["modelo"].predict(
                entradas[paquete["variables"]]
            ),
            temperatura_historial_c=celdas["media_previa_c"],
            version_modelo=paquete["version"],
        )
    )

por_fecha = pd.concat(predicciones, ignore_index=True)
promedio = (
    por_fecha.groupby(["id_celda", "col_celda", "fila_celda"], as_index=False)
    .agg(promedio_estimado_c=("temperatura_estimada_c", "mean"))
)

# Residuos de 2026 del Ridge entrenado solo con 2023–2025.
residuos = test_temp[[
    "id_celda", "fecha", "temperatura_mediana_c", "pred_ridge_c"
]].copy()
residuos["id_celda"] = residuos["id_celda"].astype("string")
residuos["fecha"] = pd.to_datetime(residuos["fecha"])
residuos["error_c"] = residuos["temperatura_mediana_c"] - residuos["pred_ridge_c"]

errores = residuos.pivot(
    index="id_celda", columns="fecha", values="error_c"
).sort_index(axis=1)

if errores.shape[1] != 6:
    raise ValueError("Se esperaban las seis fechas de evaluación de 2026.")

ventanas = pd.DataFrame({
    f"ventana_{i + 1}": errores.iloc[:, i:i + 4].mean(axis=1, skipna=False)
    for i in range(3)
})

# Exigimos las tres ventanas completas para dibujar la banda.
completas = ventanas.dropna()
banda = completas.agg(["min", "max"], axis=1).rename(columns={
    "min": "error_promedio_min_c",
    "max": "error_promedio_max_c",
})

resultado = promedio.merge(
    banda, left_on="id_celda", right_index=True, how="left", validate="one_to_one"
)
resultado["limite_inferior_exploratorio_c"] = (
    resultado["promedio_estimado_c"] + resultado["error_promedio_min_c"]
)
resultado["limite_superior_exploratorio_c"] = (
    resultado["promedio_estimado_c"] + resultado["error_promedio_max_c"]
)
resultado["ancho_banda_c"] = (
    resultado["limite_superior_exploratorio_c"]
    - resultado["limite_inferior_exploratorio_c"]
)

salida = carpeta / "promedio_multifecha_demostracion"
salida.mkdir(exist_ok=True)
por_fecha.to_csv(salida / "predicciones_por_fecha.csv", index=False)
resultado.to_csv(salida / "promedio_y_banda_exploratoria.csv", index=False)

geometrias = [
    box(col * 250, fila * 250, (col + 1) * 250, (fila + 1) * 250)
    for col, fila in zip(resultado["col_celda"], resultado["fila_celda"])
]
mapa = gpd.GeoDataFrame(resultado, geometry=geometrias, crs="EPSG:32616")

fig, ejes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
mapa.plot(
    ax=ejes[0], column="promedio_estimado_c", cmap="inferno", legend=True,
    legend_kwds={"label": "Temperatura superficial (°C)", "shrink": 0.75},
)
ejes[0].set_title("Promedio estimado de cuatro fechas")

mapa.plot(
    ax=ejes[1], column="ancho_banda_c", cmap="viridis", legend=True,
    missing_kwds={"color": "#dddddd"},
    legend_kwds={"label": "Ancho de la banda (°C)", "shrink": 0.75},
)
ejes[1].set_title("Banda exploratoria · Gris: sin historial completo")

for ax in ejes:
    ax.set_axis_off()

fig.suptitle("Managua · Temperatura superficial · Demostración 2027")
fig.savefig(salida / "mapa_promedio_y_banda.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Celdas con promedio: {len(resultado):,}")
print(f"Celdas con banda exploratoria: {resultado['ancho_banda_c'].notna().sum():,}")
print("Archivos:", salida.resolve())

In [ ]:
n_fechas = por_fecha.groupby("id_celda")["fecha_objetivo"].nunique()
assert n_fechas.eq(4).all(), "Hay celdas sin las cuatro predicciones."

con_banda = resultado["ancho_banda_c"].notna()
anchos = resultado.loc[con_banda, "ancho_banda_c"]

print(f"Celdas con cuatro predicciones: {len(n_fechas):,}")
print(f"Celdas con banda: {con_banda.sum():,} de {len(resultado):,} ({con_banda.mean():.1%})")
print(f"Ancho mediano de la banda: {anchos.median():.2f} °C")
print(f"P90 del ancho de la banda: {anchos.quantile(0.90):.2f} °C")